In [4]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns


df= pd.read_csv(r"C:\Users\user\Desktop\scraping\scraped_data.csv", encoding="utf-8-sig")


In [5]:
df

,titre,prix,ville,surface,chambres,salle_de_bain,lien
0,Appartement à vendre 154 m² à Marrakech,2 475 000 DH,"Appartements dans Marrakech, Route de Casablanca",154 m²,3 chambres,3 sdbs,https://www.avito.ma/fr/route_de_casablanca/ap...
1,Appartement à vendre 140 m² la marina Casablanca,3 900 000 DH,"Appartements dans Casablanca, Marina",155 m²,2 chambres,3 sdbs,https://www.avito.ma/fr/marina/appartements/Ap...
2,Appartement à vendre 61 m² à Marrakech,1 530 000 DH,"Appartements dans Marrakech, Guéliz",61 m²,1 chambre,1 sdb,https://www.avito.ma/fr/gu%C3%A9liz/appartemen...
3,Appartement à vendre à Temara,960 000 DH,"Appartements dans Temara, Autre secteur",128 m²,3 chambres,1 sdb,https://www.avito.ma/fr/autre_secteur/appartem...
4,Appartement en vente 61m² à Gueliz Piscine Roo...,1 500 000 DH,"Appartements dans Marrakech, Guéliz",61 m²,1 chambre,1 sdb,https://www.avito.ma/fr/gu%C3%A9liz/appartemen...
...,...,...,...,...,...,...,...
826,Nour Confort Targa,1 024 600 DH,Immobilier neuf dans Casablanca,NaN,NaN,NaN,https://immoneuf.avito.ma/fr/unite/UIH?utm_sou...
827,Joli Appartement 1er Etage Mhamid 7,410 000 DH,"Appartements dans Marrakech, M'Hamid",57 m²,2 chambres,1 sdb,https://www.avito.ma/fr/m_hamid/appartements/J...
828,"🌊 appartement vue mer en face, sans vis-à-vis",1 600 000 DH,"Appartements dans Agadir, Anza",80 m²,2 chambres,2 sdbs,https://www.avito.ma/fr/anza/appartements/___a...
829,Rez-de-jardin à vendre 128 m² 20 m² jardin,2 100 000 DH,"Appartements dans Rabat, Souissi",148 m²,3 chambres,2 sdbs,https://www.avito.ma/fr/souissi/appartements/R...


In [40]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   titre          0 non-null      object 
 1   prix           0 non-null      int64  
 2   ville          0 non-null      object 
 3   surface        0 non-null      int64  
 4   chambres       0 non-null      float64
 5   salle_de_bain  0 non-null      float64
 6   lien           0 non-null      object 
 7   prix_par_m2    0 non-null      float64
dtypes: float64(3), int64(2), object(3)
memory usage: 0.0+ bytes


# Nettoyage des données (Clean Layer)

In [ ]:



# 1. PRIX CLEANING

df["prix"] = (
    df["prix"]
    .astype(str) 
    .str.replace(r"[^\d]", "", regex=True)
)

df["prix"] = pd.to_numeric(df["prix"], errors="coerce")


# 2. SURFACE CLEANING

df["surface"] = (
    df["surface"]
    .astype(str)
    .str.replace(r"[^\d]", "", regex=True)
)
df["surface"] = pd.to_numeric(df["surface"], errors="coerce")


# 3. NUMERIC COLUMNS

df["chambres"] = pd.to_numeric(df["chambres"], errors="coerce")
df["salle_de_bain"] = pd.to_numeric(df["salle_de_bain"], errors="coerce")


# 4. CATEGORICAL

df["ville"] = df["ville"].astype("category")

# 5. CLEAN DATA

df = df.dropna(subset=["prix", "surface", "ville"])


# 6. CHECK TYPES

print(df.dtypes)

titre              object
prix                int64
ville            category
surface             int64
chambres          float64
salle_de_bain     float64
lien               object
dtype: object


In [13]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   titre          0 non-null      object  
 1   prix           0 non-null      int64   
 2   ville          0 non-null      category
 3   surface        0 non-null      int64   
 4   chambres       0 non-null      float64 
 5   salle_de_bain  0 non-null      float64 
 6   lien           0 non-null      object  
 7   prix_par_m2    0 non-null      float64 
dtypes: category(1), float64(3), int64(2), object(2)
memory usage: 9.8+ KB


,prix,surface,chambres,salle_de_bain,prix_par_m2
count,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN


# Feature Engineering

In [30]:

from datetime import datetime


# FUNCTION: FEATURE ENGINEERING

def feature_engineering(df):
    
    df = df.copy()


    # 1. BASIC CLEAN (safety)

    df = df.dropna(subset=["prix", "surface", "ville"])
    df = df[df["surface"] > 0]

   
    # 2. PRIX PAR M2
   
    df["prix_par_m2"] = df["prix"] / df["surface"]

  
    # 3. AGE DU BIEN (optional)
  
    if "annee_construction" in df.columns:
        current_year = datetime.now().year
        df["age_bien"] = current_year - df["annee_construction"]
        df["age_bien"] = df["age_bien"].where(df["age_bien"] >= 0)

    # 4. NORMALISATION GEO
   
    df["ville"] = df["ville"].astype(str).str.strip().str.lower()

    if "quartier" in df.columns:
        df["quartier"] = df["quartier"].astype(str).str.strip().str.lower()

   
    # 5. INDICATEURS GEO
   
    df["nb_annonces_ville"] = df.groupby("ville")["ville"].transform("count")

    if "quartier" in df.columns:
        df["nb_annonces_quartier"] = df.groupby("quartier")["quartier"].transform("count")

    
    # 6. SEGMENTATION PRIX
    df["segment_prix"] = pd.qcut(
    df["prix"],
    q=3,
    labels=["low", "medium", "high"],
    duplicates="drop"
)

    # 7. SEGMENTATION SURFACE
    
    df["segment_surface"] = pd.cut(
        df["surface"],
        bins=[0, 50, 100, 200, 1000],
        labels=["small", "medium", "large", "luxury"]
    )

   
    # 8. FEATURE METIER
    
    df["valeur_estimee"] = df["prix_par_m2"] * df["surface"]

    
    # 9. OUTLIERS FILTER (optional)
    
    df = df[(df["prix"] > 0) & (df["prix"] < 1e7)]
    df = df[(df["surface"] > 5) & (df["surface"] < 1000)]

    
    # 10. FINAL CLEAN
   
    df = df.dropna()

    return df





In [33]:
print(df.columns)
print(df.head())

Index(['titre', 'prix', 'ville', 'surface', 'chambres', 'salle_de_bain',
       'lien', 'prix_par_m2'],
      dtype='object')
Empty DataFrame
Columns: [titre, prix, ville, surface, chambres, salle_de_bain, lien, prix_par_m2]
Index: []
